# 03 Density Scoring

**Description:** Convert lane assignment counts into density scores for adaptive traffic decisions.

**Objective:** Aggregate counts by lane, compute density scores, and save both CSV and plot outputs into `outputs/density_scoring/`.

## Step 1. Setup and Input Loading

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Outputs root: {OUTPUT_ROOT}')

OUTPUT_DIR = OUTPUT_ROOT / 'density_scoring'
os.makedirs(OUTPUT_DIR, exist_ok=True)
INPUT_CSV = OUTPUT_ROOT / 'lane_assignment' / 'lane_assignments.csv'
DENSITY_CSV = OUTPUT_DIR / 'density_scores.csv'
DENSITY_PLOT = OUTPUT_DIR / 'density_scores.png'


In [ ]:
if INPUT_CSV.exists():
    lane_df = pd.read_csv(INPUT_CSV)
else:
    lane_df = pd.DataFrame({'lane_id': ['lane_1', 'lane_2', 'lane_2', 'lane_3', 'lane_4']})

lane_df.head()


## Step 2. Density Calculation

This skeleton uses a simple normalized count-based density. Replace the formula with occupancy, queue length, or speed-aware scoring if needed.

In [ ]:
def compute_density_scores(df: pd.DataFrame, capacity: int = 20) -> pd.DataFrame:
    counts = df.groupby('lane_id').size().rename('vehicle_count').reset_index()
    counts['density_score'] = counts['vehicle_count'] / capacity
    return counts.sort_values('lane_id').reset_index(drop=True)

density_df = compute_density_scores(lane_df)
density_df.to_csv(DENSITY_CSV, index=False)
print(f'Saved density scores to {DENSITY_CSV}')
density_df


## Step 3. Save Density Plot

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(density_df['lane_id'], density_df['density_score'], color='steelblue')
plt.title('Lane Density Scores')
plt.xlabel('Lane')
plt.ylabel('Density Score')
plt.tight_layout()
plt.savefig(DENSITY_PLOT, dpi=150)
plt.show()
print(f'Saved plot to {DENSITY_PLOT}')
